# Stock on Hand — Order Preparation
**Foodland Wudinna | Fruit & Veg**

Run before each order cycle. Loads the POS stock export, compares against active items,
classifies stock reliability, computes a day-by-day EWMA demand forecast, and generates
the printable stock count sheet.

**Pipeline position:**
1. Run `parse_soh_export.py` → generates `stock_on_hand_v2.csv`
2. Run this notebook → generates `StockCountSheet_YYYYMMDD.xlsx` (print & count)
3. Fill in manual stock counts on the printed sheet
4. Update `stock_on_hand_v2.csv` with manual counts
5. Run `FruitVeg_Demand_Forecast.ipynb` → generates the order list


## Section 1 — Configuration
Edit this section before each run.

In [1]:
from pathlib import Path
import pandas as pd, numpy as np, re
from datetime import date

ROOT = Path('../').resolve()  # = foodland_wudinna/

# ── POS stock export — auto-detects the most recent Stock_*.xlsx ─────────────
_soh_files = sorted((ROOT / '01_data/operational').glob('Stock_*.xlsx'), reverse=True)
if not _soh_files:
    raise FileNotFoundError("No Stock_*.xlsx found in 01_data/operational/. Export from POS first.")
SOH_FILE = _soh_files[0]
print(f"Using SOH export: {SOH_FILE.name}")

# ── Order type ─────────────────────────────────────────────────────────
ORDER_TYPE = 'WED_FRI'   # 'WED_FRI' or 'FRI_TUE'

# ── Cycle trading days this order needs to cover ───────────────────────
# These are the store-open days AFTER the delivery, up to the next delivery day.
# Cross-check against dim_calendar.csv (is_store_open = 1).
CYCLE_DATES = ['2026-04-07', '2026-04-08', '2026-04-09', '2026-04-10']

# ── Lookback for EWMA forecast ─────────────────────────────────────────
LOOKBACK_WEEKS      = 8     # weeks of recent sales to use
STOCK_ERROR_THRESHOLD = -50 # stock values below this are POS system errors
EWMA_ALPHA          = 0.35  # matches FruitVeg_Demand_Forecast.ipynb

def norm(s): return re.sub(r'\s+', ' ', str(s)).strip()

cycle_dates  = [pd.Timestamp(d) for d in CYCLE_DATES]
cycle_labels = [d.strftime('%a %d/%m') for d in cycle_dates]

print('Order type :', ORDER_TYPE)
print('Cycle      :', ', '.join(cycle_labels))
print('EWMA alpha :', EWMA_ALPHA)


Order type : WED_FRI
Cycle      : Tue 07/04, Wed 08/04, Thu 09/04, Fri 10/04


## Section 2 — Load Stock On Hand from POS Export

In [2]:
raw = pd.read_excel(SOH_FILE, sheet_name='Page 1', header=5)
raw.columns = [re.sub(r'\s+', ' ', str(c)).strip() for c in raw.columns]
raw = raw.rename(columns={'Description':'Name','Stock On Hand':'Stock',
                           'Last Sold':'Last_Sold','Weeks On Hand':'Weeks_on_Hand'})
raw['Name']  = raw['Name'].apply(norm)
raw['Stock'] = pd.to_numeric(raw['Stock'], errors='coerce').fillna(0)
raw['AWS']   = pd.to_numeric(raw['AWS'],   errors='coerce').fillna(0)
soh = raw[raw['Name'].str.len() > 2][['Name','Stock','AWS','Weeks_on_Hand','Last_Sold']].copy().reset_index(drop=True)
print(f'SOH items : {len(soh)}')
print(f'  Stock > 0 : {(soh["Stock"] > 0).sum()}')
print(f'  Stock = 0 : {(soh["Stock"] == 0).sum()}')
print(f'  Stock < 0 : {(soh["Stock"] < 0).sum()}  <- includes system errors')
soh.head()

SOH items : 144
  Stock > 0 : 57
  Stock = 0 : 49
  Stock < 0 : 38  <- includes system errors


,Name,Stock,AWS,Weeks_on_Hand,Last_Sold
0,TURNIPS LOOSE PER KG,-479551.737,1146.1000,-418.420502,21 Mar 26
1,TOMATOES - RAINBOW TOMATO MEDLEY 250GMS,2.000,4.5889,0.435834,25 Mar 26
2,TOMATOES - CHERRY SAMPARI TRUSS 250G,4.000,13.1444,0.304312,26 Mar 26
3,TOMATOES - CHERRY PUNNET 250G,8.000,30.8000,0.25974,26 Mar 26
4,SWEETCORN PREPACK,-3941.000,0.0000,0,24 Feb 26


## Section 3 — Load Active Items from Recent Sales

In [3]:
sales = pd.read_csv(ROOT / '01_data/raw/sales_fruit_2026.csv', low_memory=False)
sales.columns = sales.columns.str.strip()
sales['Name']     = sales['Name'].apply(norm)
sales['Date']     = pd.to_datetime(sales['Date'], errors='coerce').dt.normalize()
sales['Quantity'] = pd.to_numeric(sales['Quantity'],     errors='coerce').fillna(0)
sales['Revenue']  = pd.to_numeric(sales['Sales Ex GST'], errors='coerce').fillna(0)
sales = sales[~sales['Name'].str.upper().str.contains('FRUIT AND VEG|REDUCED', na=False)]

cutoff = sales['Date'].max() - pd.Timedelta(weeks=LOOKBACK_WEEKS)
recent = sales[sales['Date'] >= cutoff].copy()
recent['dow'] = recent['Date'].dt.dayofweek

# Sub-department: take the most common value per item
subdept_col = 'Sub Department Name' if 'Sub Department Name' in sales.columns else None

agg_spec = {'Revenue': ('Revenue', 'sum'), 'Qty': ('Quantity', 'sum'), 'Days': ('Date', 'nunique')}
if subdept_col:
    agg_spec['SubDept'] = (subdept_col, lambda x: x.mode().iloc[0] if len(x) > 0 else 'Other')

active = (
    recent.groupby('Name')
    .agg(**agg_spec)
    .sort_values('Revenue', ascending=False)
    .reset_index()
)
if 'SubDept' not in active.columns:
    active['SubDept'] = 'Other'

print(f'Active items (last {LOOKBACK_WEEKS} weeks): {len(active)}')
print(f'Period: {recent["Date"].min().date()} to {recent["Date"].max().date()}')
print(f'Sub-departments: {active["SubDept"].nunique()} — {sorted(active["SubDept"].unique()[:5])}...')


Active items (last 8 weeks): 214
Period: 2026-01-22 to 2026-03-19


## Section 4 — Compare Active Items vs SOH

In [4]:
merged = active.merge(soh, on='Name', how='left')

def classify(row):
    if pd.isna(row['Stock']): return 'Missing from SOH'
    if row['Stock'] < STOCK_ERROR_THRESHOLD: return 'Invalid (system error)'
    if row['Stock'] < 0: return 'Slightly negative'
    if row['Stock'] == 0: return 'Stock = 0'
    return 'Valid stock'

merged['Stock_Status'] = merged.apply(classify, axis=1)
print('=== STOCK STATUS SUMMARY ===')
print(merged['Stock_Status'].value_counts().to_string())
print()
print('Missing from SOH    = loose/weighed items — POS cannot track by unit')
print('Invalid system error = POS accumulated weight scan errors — treat as 0')
print('Both groups require a physical count before placing the order.')

=== STOCK STATUS SUMMARY ===
Stock_Status
Missing from SOH          134
Valid stock                49
Invalid (system error)     27
Stock = 0                   3
Slightly negative           2

Missing from SOH    = loose/weighed items — POS cannot track by unit
Invalid system error = POS accumulated weight scan errors — treat as 0
Both groups require a physical count before placing the order.


### Items with valid stock (ready to use)

In [5]:
valid = merged[merged['Stock_Status'] == 'Valid stock'].copy()
print(f'{len(valid)} items with usable system stock:')
valid[['Name','Stock','AWS','Revenue','Qty']].sort_values('Revenue', ascending=False)

49 items with usable system stock:


,Name,Stock,AWS,Revenue,Qty
6,BLUEBERRIES 125G,21.0,52.1889,2921.510,419.0
10,POTATOES - WASHED 2.5KG BAG,14.0,37.9556,2108.990,330.0
12,R/BOW FRESH BABY SPINACH 120G,19.0,57.9444,1869.720,511.0
14,CARROTS 1 KG,38.0,85.8667,1788.700,726.0
17,SNOW PEAS PRE PACK 150G,19.0,27.3778,1543.300,234.0
19,CUCUMBER BABY LEBANESE P/PACK 250GMS,32.0,45.1111,1502.510,404.0
39,HERBS - FRESH ASSTD VARIETIES,4.0,25.1222,773.940,195.0
41,LK PASTA SALAD 800G,3.0,13.6889,758.420,104.0
43,R/BOW FRESH MESCULIN MIX 120GMS,16.0,26.2889,720.300,210.0
47,R/BOW FRESH SWEET MIX 120GMS,1.0,24.8889,678.180,194.0


### Items needing a manual stock count

In [6]:
manual_needed = merged[merged['Stock_Status'].isin(
    ['Missing from SOH','Invalid (system error)','Slightly negative','Stock = 0'])].copy()
print(f'{len(manual_needed)} items need a physical count before ordering:')
manual_needed[['Name','Stock','Stock_Status','Revenue','Qty']].sort_values('Revenue', ascending=False)

166 items need a physical count before ordering:


,Name,Stock,Stock_Status,Revenue,Qty
0,BANANAS PER KG,NaN,Missing from SOH,9613.57,1643.179
1,STRAWBERRIES PER PUNNET,0.000,Stock = 0,6791.94,1066.000
2,LETTUCE PER EACH,NaN,Missing from SOH,4312.60,1164.000
3,WATERMELON SEEDLESS PER KG,NaN,Missing from SOH,3934.73,1674.758
4,APPLES PINK LADY PER KG,NaN,Missing from SOH,3729.73,449.560
...,...,...,...,...,...
210,APPLES - FUJI PER KG,NaN,Missing from SOH,4.56,0.761
211,APPLES JONATHON PER KG,-45155.513,Invalid (system error),3.48,0.634
212,BANANAS RIPE PER KG,NaN,Missing from SOH,3.08,1.029
213,COMM CO SALAD CRMY RNCH 350GM,NaN,Missing from SOH,3.00,2.000


## Section 5 — EWMA Forecast per Cycle Day

Uses the same EWMA (α = 0.35) algorithm as `FruitVeg_Demand_Forecast.ipynb`.
For each item and each trading day in the cycle, the forecast is the exponentially-weighted
average of the last 6 occurrences of that weekday in recent sales.
Items with no same-DOW history fall back to their overall daily average.


In [ ]:
alpha = EWMA_ALPHA
forecast = pd.DataFrame({'Name': merged['Name'].tolist()})

for tgt_date, label in zip(cycle_dates, cycle_labels):
    dow = tgt_date.dayofweek
    day_forecasts = []

    for item in merged['Name']:
        item_df = recent[recent['Name'] == item]
        same_dow = item_df[item_df['dow'] == dow].sort_values('Date')
        qtys = same_dow['Quantity'].values

        if len(qtys) == 0:
            # No same-DOW history — use overall average
            fallback = item_df['Quantity'].mean() if len(item_df) else 0.0
            day_forecasts.append(round(fallback, 1))
            continue

        # EWMA over last 6 same-DOW observations (most recent first)
        lags = qtys[-6:][::-1]
        weights = np.array([alpha ** i for i in range(len(lags))])
        weights /= weights.sum()
        ewma_val = float(np.dot(weights, lags))
        day_forecasts.append(round(ewma_val, 1))

    forecast[label] = day_forecasts

forecast['Total_Forecast'] = forecast[cycle_labels].sum(axis=1).round(1)

print(f'Forecast ready: {len(forecast)} items × {len(cycle_dates)} trading days')
print(f'Cycle total demand: {forecast["Total_Forecast"].sum():.0f} units across all items')
forecast.head(5)


## Section 6 — Extended Stock Count Sheet

Generates a print-ready Excel file with:
- One column per cycle trading day (EWMA forecast)
- Total Forecast column
- SOH column: pre-filled for system-tracked items, blank for manual
- Order Qty = max(Total Forecast − SOH, 0): pre-filled for system-tracked items, blank for manual
- Notes column

**Row colours:**  🔵 Blue = system-tracked (SOH pre-filled)  ⬜ White/grey = needs manual count


In [ ]:
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter
from openpyxl.worksheet.page import PageMargins

# ── Assemble the working dataframe ─────────────────────────────────────
sheet_df = forecast.merge(
    merged[['Name', 'Stock', 'Stock_Status']].rename(columns={'Stock': 'Sys_Stock'}),
    on='Name', how='left'
).merge(
    active[['Name', 'SubDept', 'Revenue']],
    on='Name', how='left'
)
sheet_df['SubDept'] = sheet_df['SubDept'].fillna('Other')

# Mark system-tracked items (reliable stock in POS)
sheet_df['is_system'] = sheet_df['Stock_Status'] == 'Valid stock'

# Computed order qty for system-tracked items only
def calc_order_qty(row):
    if not row['is_system']:
        return np.nan
    soh_v = row['Sys_Stock'] if not pd.isna(row['Sys_Stock']) else 0
    return max(0, round(row['Total_Forecast'] - soh_v))

sheet_df['Order_Qty'] = sheet_df.apply(calc_order_qty, axis=1)

# Sort: sub-dept alphabetically, then by revenue descending within each sub-dept
sheet_df = sheet_df.sort_values(['SubDept', 'Name'], ascending=[True, True]).reset_index(drop=True)

# ── Colour palette ─────────────────────────────────────────────────────
TEAL_DARK  = 'C0392B'; TEAL_DARK  = '1A5276'   # section header bg
BLUE_ITEM  = 'D6EAF8'                            # system-tracked rows
GREY_ITEM  = 'F2F3F4'                            # manual rows (even)
WHITE_ITEM = 'FFFFFF'                            # manual rows (odd)
HEADER_BG  = '1A5276'                            # column header bg
FORECAST_COL_BG = 'EBF5FB'                      # day forecast columns
TOTAL_COL_BG    = 'D5E8D4'                      # total forecast column
ORDER_COL_BG    = 'FCF3CF'                      # order qty column

def make_fill(hex_colour):
    return PatternFill('solid', fgColor=hex_colour)

thin_border = Border(
    left=Side('thin', color='CCCCCC'), right=Side('thin', color='CCCCCC'),
    top=Side('thin', color='CCCCCC'), bottom=Side('thin', color='CCCCCC'),
)
center = Alignment(horizontal='center', vertical='center', wrap_text=False)
left_al = Alignment(horizontal='left', vertical='center', wrap_text=False)

# ── Column layout ──────────────────────────────────────────────────────
# Col 1: #   Col 2: Item   Col 3..N: forecast days   N+1: Total   N+2: SOH   N+3: Order Qty   N+4: Notes
n_days  = len(cycle_labels)
COL_NO    = 1
COL_NAME  = 2
COL_DAYS  = list(range(3, 3 + n_days))           # e.g. [3,4,5,6] for 4 days
COL_TOTAL = 3 + n_days
COL_SOH   = 4 + n_days
COL_ORDER = 5 + n_days
COL_NOTES = 6 + n_days
TOTAL_COLS = COL_NOTES

# ── Column widths ──────────────────────────────────────────────────────
wb = Workbook()
ws = wb.active
ws.title = 'Stock Count'

ws.column_dimensions['A'].width = 4    # #
ws.column_dimensions['B'].width = 36   # Item Name
for col_i in COL_DAYS:
    ws.column_dimensions[get_column_letter(col_i)].width = 8
ws.column_dimensions[get_column_letter(COL_TOTAL)].width = 9
ws.column_dimensions[get_column_letter(COL_SOH)].width   = 8
ws.column_dimensions[get_column_letter(COL_ORDER)].width  = 9
ws.column_dimensions[get_column_letter(COL_NOTES)].width  = 20

# ── Row 1: Title ───────────────────────────────────────────────────────
ws.row_dimensions[1].height = 22
ws.merge_cells(f'A1:{get_column_letter(TOTAL_COLS)}1')
title_cell = ws['A1']
title_cell.value = f'STOCK COUNT SHEET — Foodland Wudinna | Fruit & Veg'
title_cell.font  = Font('Arial', 14, bold=True, color='FFFFFF')
title_cell.fill  = make_fill(HEADER_BG)
title_cell.alignment = Alignment(horizontal='center', vertical='center')

# ── Row 2: Cycle info ──────────────────────────────────────────────────
ws.row_dimensions[2].height = 16
ws.merge_cells(f'A2:{get_column_letter(TOTAL_COLS)}2')
order_date_str = date.today().strftime('%d %b %Y')
cycle_str = ', '.join([f'{d.strftime("%a %d %b")}' for d in cycle_dates])
ws['A2'].value = f'Order date: {order_date_str}  |  Cycle covers: {cycle_str}  |  Order type: {ORDER_TYPE}'
ws['A2'].font  = Font('Arial', 9, color='444444')
ws['A2'].fill  = make_fill('EAF2FF')
ws['A2'].alignment = Alignment(horizontal='center', vertical='center')

# ── Row 3: Legend ──────────────────────────────────────────────────────
ws.row_dimensions[3].height = 14
ws.merge_cells(f'A3:{get_column_letter(TOTAL_COLS)}3')
ws['A3'].value = '🔵 Blue = system-tracked (SOH pre-filled, Order Qty calculated)   ⬜ White/Grey = count manually'
ws['A3'].font  = Font('Arial', 8, italic=True, color='555555')
ws['A3'].fill  = make_fill('FDFEFE')
ws['A3'].alignment = Alignment(horizontal='left', vertical='center')

# ── Row 4: Column headers ──────────────────────────────────────────────
ws.row_dimensions[4].height = 30
hdr_font = Font('Arial', 9, bold=True, color='FFFFFF')
hdr_fill = make_fill(HEADER_BG)

headers = ['#', 'Item Name'] + cycle_labels + ['Total\nForecast', 'SOH', 'Order\nQty', 'Notes']
for col_i, hdr_text in enumerate(headers, 1):
    c = ws.cell(4, col_i, hdr_text)
    c.font = hdr_font
    c.fill = hdr_fill
    c.alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)
    c.border = thin_border

# ── Body rows ──────────────────────────────────────────────────────────
row_num  = 5
item_num = 0
current_subdept = None

for _, item_row in sheet_df.iterrows():
    subdept = item_row['SubDept']

    # Sub-department section header
    if subdept != current_subdept:
        current_subdept = subdept
        ws.row_dimensions[row_num].height = 16
        ws.merge_cells(f'A{row_num}:{get_column_letter(TOTAL_COLS)}{row_num}')
        sec_cell = ws.cell(row_num, 1, subdept.upper())
        sec_cell.font  = Font('Arial', 10, bold=True, color='FFFFFF')
        sec_cell.fill  = make_fill('1A5276')
        sec_cell.alignment = Alignment(horizontal='left', vertical='center', indent=1)
        for col_i in range(1, TOTAL_COLS + 1):
            ws.cell(row_num, col_i).border = thin_border
        row_num += 1

    # Item row
    item_num += 1
    is_system = bool(item_row['is_system'])
    row_fill  = make_fill(BLUE_ITEM) if is_system else (make_fill(GREY_ITEM) if row_num % 2 == 0 else make_fill(WHITE_ITEM))

    ws.row_dimensions[row_num].height = 16

    # Col 1: #
    c = ws.cell(row_num, COL_NO, item_num)
    c.font = Font('Arial', 8, color='888888')
    c.fill = row_fill; c.border = thin_border; c.alignment = center

    # Col 2: Item Name
    c = ws.cell(row_num, COL_NAME, item_row['Name'])
    c.font = Font('Arial', 9, bold=is_system)
    c.fill = row_fill; c.border = thin_border; c.alignment = left_al

    # Forecast day columns
    for col_i, label in zip(COL_DAYS, cycle_labels):
        val = item_row.get(label, 0)
        c = ws.cell(row_num, col_i, round(float(val), 1))
        c.font = Font('Arial', 9)
        c.fill = make_fill(BLUE_ITEM) if is_system else make_fill(FORECAST_COL_BG)
        c.border = thin_border; c.alignment = center
        c.number_format = '0.0'

    # Total Forecast
    c = ws.cell(row_num, COL_TOTAL, round(float(item_row['Total_Forecast']), 1))
    c.font = Font('Arial', 9, bold=True)
    c.fill = make_fill(TOTAL_COL_BG); c.border = thin_border; c.alignment = center
    c.number_format = '0.0'

    # SOH column
    if is_system:
        soh_val = int(item_row['Sys_Stock']) if not pd.isna(item_row['Sys_Stock']) else 0
        c = ws.cell(row_num, COL_SOH, soh_val)
        c.font = Font('Arial', 9, bold=True, color='1A5276')
    else:
        c = ws.cell(row_num, COL_SOH, '')  # blank — write manually
        c.font = Font('Arial', 9)
    c.fill = make_fill(BLUE_ITEM) if is_system else make_fill(WHITE_ITEM)
    c.border = thin_border; c.alignment = center

    # Order Qty column
    if is_system:
        ord_val = int(item_row['Order_Qty']) if not pd.isna(item_row['Order_Qty']) else 0
        c = ws.cell(row_num, COL_ORDER, ord_val)
        c.font = Font('Arial', 9, bold=True, color='C0392B' if ord_val > 0 else '27AE60')
    else:
        c = ws.cell(row_num, COL_ORDER, '')  # blank — calculate manually
        c.font = Font('Arial', 9)
    c.fill = make_fill(ORDER_COL_BG); c.border = thin_border; c.alignment = center

    # Notes column (blank)
    c = ws.cell(row_num, COL_NOTES, '')
    c.fill = make_fill(WHITE_ITEM); c.border = thin_border; c.alignment = left_al

    row_num += 1

# ── Print settings ─────────────────────────────────────────────────────
ws.print_title_rows = '1:4'
ws.page_setup.orientation = 'landscape'
ws.page_setup.paperSize   = 9   # A4
ws.page_setup.fitToPage   = True
ws.page_setup.fitToWidth  = 1
ws.page_setup.fitToHeight = 0
ws.page_margins = PageMargins(left=0.4, right=0.4, top=0.6, bottom=0.6)
ws.freeze_panes = 'C5'

# ── Save ───────────────────────────────────────────────────────────────
fname = f'StockCountSheet_{date.today().strftime("%Y%m%d")}.xlsx'
out_path = ROOT / '04_ordering' / fname
wb.save(str(out_path))

print(f'Stock count sheet saved: {fname}')
sys_items = sheet_df['is_system'].sum()
man_items = (~sheet_df['is_system']).sum()
print(f'  System-tracked rows (SOH pre-filled) : {sys_items}')
print(f'  Manual count rows (SOH blank)         : {man_items}')
print(f'  Total items                           : {len(sheet_df)}')
print(f'  Cycle total forecast                  : {sheet_df["Total_Forecast"].sum():.0f} units')
print(f'  Pre-calculated order qty (system only): {sheet_df["Order_Qty"].sum():.0f} units')
